# N=48 leak-free: the depth ladder closes, and L\*(T) rises with reasoning depth

**Jobs `50527633` (L=12, the anchor) and `50527634` (L=8), submitted 2026-10-31 12:03, commit `4b9c0e5`, seed 0.**

Finding 33 (`50099945`, 10-29) ran `num_layers` in {6, 4, 3} at N=48 on the **leak-free** task and found
**no rung that clears the 0.90 gate**: L\*(48, leak-free, ff=1024) was *undefined above 6*, not large. This
pair of jobs goes **upward** instead, adding L=8 and L=12 on the same instances, the same budget
(240000 x 100ep) and the same shared pool, with the 10-29 cells re-evaluated inside each job so every
contrast is paired.

**Headline.** At the harder density (rho=0.60, six depth bins) the gate is cleared in **0 of 6 bins at L=6,
3 of 6 at L=8 and 6 of 6 at L=12**, so **L\*(48, leak-free, ff=1024, rho=0.60) lies in (8, 12]** -- the ladder
is closed. The three bins L=8 misses are **the three deepest**, so its gate crosses between T=5 and T=6. This
is the second cell ever measured *on* the gate (after finding 22's N=96 L=6) and the **first in the leak-free
task**, where no adjacency-order shortcut (finding 16) can be doing the work.

**L\* is a function of density too, and this run shows it inside one job.** At rho=0.75, where the deepest bin
is T=4 instead of T=8, **L=8 clears all three bins and L\*=8**. Every L\* in this project must therefore be
read with its density attached, the same way finding 14 showed it must be read with its MLP width attached.
The second thing worth noticing is that **L=12 clears every bin and still falls**, 1.000 at T=3 to 0.957 at
T=8 -- above the line, not flat, so the depth demand has not saturated at twelve layers either.

In [ ]:
import glob, io, json, math, os, re
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt

S = os.environ["SCRATCH"] + "/spin_gnn"
DIRS = [f"{S}/sudoku_n48_leakfree_lstar_2026-10-29",   # 50099945: L=6, 4, 3
        f"{S}/sudoku_n48_leakfree_lstar_2026-10-31"]   # 50527633: L=12, 50527634: L=8
GATE, LADDER = 0.90, [12, 8, 6, 4, 3]

lab = lambda c: "L=%d" % int(re.search(r"_L(\d+)_", c).group(1))
depth_of = lambda c: int(re.search(r"_L(\d+)_", c).group(1))


def wilson(k, n, z=1.96):
    p, d = k / n, 1 + z * z / n
    c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d


def mcnemar_exact(b, c):
    # two-sided exact McNemar on the discordant counts
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)


def ca_trend(ks, ns, xs):
    # Cochran-Armitage trend z over ordered bins; negative = falling with x
    ks, ns, xs = np.asarray(ks, float), np.asarray(ns, float), np.asarray(xs, float)
    N, K = ns.sum(), ks.sum()
    pbar, xbar = K / N, (ns * xs).sum() / N
    var = pbar * (1 - pbar) * (ns * (xs - xbar) ** 2).sum()
    return (ks * (xs - xbar)).sum() / np.sqrt(var) if var > 0 else float("nan")


pool_csvs, bin_csvs, npzs = [], [], []
for d in DIRS:
    pool_csvs += sorted(glob.glob(f"{d}/sharedpool_n1500_*.csv"))
    bin_csvs += sorted(glob.glob(f"{d}/depth_lstar_*.csv"))
    npzs += sorted(glob.glob(f"{d}/sharedpool_n1500_*_per_instance.npz"))
print("inputs:", len(pool_csvs), "pool CSVs,", len(bin_csvs), "bin CSVs,", len(npzs), "per-instance npz")

## 1. Provenance, config and seed

The only thing that differs between the five cells is `num_layers`. Everything else -- the generator, the
instance budget, the epochs, the optimizer, the seed, and `shuffle_adj=True` (the leak-free task, finding 16)
-- is held fixed, and the cell printed below is read from each checkpoint's own `config.json` rather than
from the submission script, so a silent divergence would show up here.

In [ ]:
KEYS = ["n_nodes", "degree", "n_colors", "rho", "rho_max", "hint_mode", "shuffle_adj", "shuffle_events",
        "d_model", "nhead", "num_layers", "dim_feedforward", "norm_first", "lr", "lr_schedule",
        "warmup_epochs", "batch_size", "n_train_instances", "epochs", "seed"]
rows = []
for d in DIRS:
    for cfg in sorted(glob.glob(f"{d}/L*/*/config.json")):
        c = json.load(io.open(cfg))
        rows.append({"cell": "L=%d" % c["num_layers"], **{k: c[k] for k in KEYS}})
cfg = pd.DataFrame(rows).sort_values("num_layers", ascending=False).set_index("cell")
varying = [k for k in KEYS if cfg[k].nunique() > 1]
print("columns that differ across cells:", varying, "   <- must be ['num_layers'] alone")
print()
print(cfg[KEYS].to_string())
print()
print("git commit at submit: 4b9c0e5   jobs: 50527633 (L=12, 14h), 50527634 (L=8, 10h),"
      " 50099945 (L=6/4/3, 10-29)")

## 2. The metric of record: the shared pool, n=1500

`sudoku_agg_eval` at `seed_start=5e10`, disjoint from every training set, instance *i* at `seed_start+i` so
the rows are the same instances for every cell and paired tests are legitimate. Each job re-evaluates the
10-29 reference cells, so L=6/4/3 are scored two or three times; the repeats are asserted to agree **instance
for instance** before anything is pooled, which is a free reproducibility check on the whole eval path.

In [ ]:
solved, seen_in = {}, {}
seed_starts = set()
for p in npzs:
    job = re.search(r"_(\d+)_per_instance", p).group(1)
    z = np.load(p, allow_pickle=True)
    seed_starts.add(int(z["seed_start"]))
    for i, name in enumerate(z["names"]):
        key, v = lab(str(name)), np.asarray(z["solved"][i], bool)
        if key in solved:
            ok = bool((solved[key] == v).all())
            print(f"repeat eval of {key:>5}: job {job} vs {seen_in[key]} -> "
                  f"{'IDENTICAL on all %d instances' % v.size if ok else 'DISAGREE'}")
            assert ok, f"{key} disagrees between {job} and {seen_in[key]}"
        else:
            solved[key], seen_in[key] = v, job
assert len(seed_starts) == 1, seed_starts
print("\nseed_start", seed_starts.pop())

rows = []
for L in LADDER:
    k = "L=%d" % L
    if k not in solved:
        continue
    v = solved[k]
    lo, hi = wilson(int(v.sum()), v.size)
    rows.append(dict(cell=k, L=L, n=v.size, solve_rate=v.mean(), wilson_lo=lo, wilson_hi=hi,
                     clears_gate=bool(v.mean() >= GATE and lo >= 0.88)))
pool = pd.DataFrame(rows)
print()
print(pool.round(4).to_string(index=False))

g = pd.concat([pd.read_csv(p) for p in pool_csvs], ignore_index=True)
g["cell"] = g.checkpoint.map(lab)
print("\nformat controls -- a solve-rate gap is only a reasoning gap if these are clean:")
print(g.groupby("cell")[["truncated_rate", "outcome_accuracy"]].agg(["min", "max"]).round(4).to_string())

## 3. Every paired contrast on the same 1500 instances

Exact McNemar, two-sided, on the discordant counts. `delta` is the difference in solve rate; `won-lost` are
the instances the first cell solves and the second does not, and vice versa.

In [ ]:
keys = [k for k in ("L=%d" % L for L in LADDER) if k in solved]
rows = []
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        x, y = solved[a], solved[b]
        won, lost = int((x & ~y).sum()), int((~x & y).sum())
        rows.append(dict(a=a, b=b, delta=x.mean() - y.mean(), won=won, lost=lost,
                         p=mcnemar_exact(won, lost)))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:+.4f}" if abs(v) < 2 else f"{v:.3g}"))

## 4. The finding: depth bins, the gate, and the trend in T

L\* is defined on **depth bins at fixed density**, not on the pool: for each requested density, instances are
binned by T (the number of DD rounds the instance actually needs) and a cell clears the gate only if **every**
bin is at or above 0.90. The density is held fixed inside each bin so that "deeper" cannot mean "sparser" --
the realized density printed below is the control, and a spread near zero is what makes the T comparison
meaningful.

`r(solve_rate, T)` and the Cochran-Armitage z are the trend statistics. A **floored or saturated cell cannot
trend**, which is the confound that demoted finding 11 and qualified findings 22 and 33; only a cell sitting
*on* the gate can show one, and here that cell is L=8.

In [ ]:
bins = pd.concat([pd.read_csv(p) for p in bin_csvs], ignore_index=True)
bins["L"] = bins.checkpoint.map(depth_of)
bins = bins.drop_duplicates(subset=["checkpoint", "rho_requested", "T"])
for rho in sorted(bins.rho_requested.unique(), reverse=True):
    sub = bins[bins.rho_requested == rho]
    piv = sub.pivot_table(index="T", columns="L", values="solve_rate")
    piv = piv[sorted(piv.columns, reverse=True)]
    nn = sub.pivot_table(index="T", columns="L", values="n").max(axis=1).astype(int)
    piv.columns = ["L=%d" % c for c in piv.columns]
    piv.insert(0, "n", nn)
    print(f"=== density requested {rho:.2f}, realized {sub.realized_rho.min():.6f}"
          f"-{sub.realized_rho.max():.6f}  (spread {sub.realized_rho.max()-sub.realized_rho.min():.6f})")
    print(piv.round(4).to_string())
    nb = sub["T"].nunique()
    clears = {int(L): int(((sub.L == L) & (sub.solve_rate >= GATE)).sum())
              for L in sorted(sub.L.unique(), reverse=True)}
    print(f"  bins at or above the {GATE:.2f} gate, of {nb}: {clears}")
    lstar = [L for L in sorted(sub.L.unique()) if clears[L] == nb]
    print("  L* on this density:", min(lstar) if lstar else "UNDEFINED (no rung clears every bin)")
    for L in sorted(sub.L.unique(), reverse=True):
        r = sub[sub.L == L].sort_values("T")
        z = ca_trend((r.solve_rate * r.n).round(), r.n, r["T"])
        rr = np.corrcoef(r["T"], r.solve_rate)[0, 1] if r.solve_rate.std() > 0 else float("nan")
        print(f"  L={L:<3} {r.solve_rate.max():.3f} -> {r.solve_rate.min():.3f} over "
              f"T={int(r['T'].min())}-{int(r['T'].max())},  r(solve,T) = {rr:+.3f},  "
              f"Cochran-Armitage z = {z:+.2f}")
    print()

## 5. Plots

Top: the shared pool against depth, with Wilson intervals and the gate. Bottom: the per-bin curves at each
density -- the panel that carries the finding is the harder density, where L=8's curve crosses the gate
between T=5 and T=6 while L=12 stays above it everywhere.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
t = pool.sort_values("L")
ax.errorbar(t.L, t.solve_rate, yerr=[t.solve_rate - t.wilson_lo, t.wilson_hi - t.solve_rate],
            fmt="o-", lw=2.5, ms=9, capsize=5, color="#2b6cb0", rasterized=True)
ax.axhline(GATE, color="k", ls="--", lw=2)
ax.text(t.L.min(), GATE + 0.02, "0.90 gate", fontsize=11)
ax.set_xticks(t.L); ax.set_xlabel("num_layers")
ax.set_ylabel("shared-pool solve rate (n=1500)"); ax.set_ylim(-0.05, 1.08)
ax.set_title("N=48 leak-free, ff=1024, 240000 x 100ep")
fig.tight_layout()

rhos = sorted(bins.rho_requested.unique(), reverse=True)
fig, axes = plt.subplots(1, len(rhos), figsize=(6.2 * len(rhos), 4.4))
axes = np.atleast_1d(axes)
for ax, rho in zip(axes, rhos):
    sub = bins[bins.rho_requested == rho]
    for L in sorted(sub.L.unique(), reverse=True):
        r = sub[sub.L == L].sort_values("T")
        ax.plot(r["T"], r.solve_rate, "o-", lw=2.5, ms=8, label=f"L={L}", rasterized=True)
    ax.axhline(GATE, color="k", ls="--", lw=2)
    ax.set_xticks(sorted(sub["T"].unique())); ax.set_ylim(-0.05, 1.08)
    ax.set_xlabel("T, DD rounds the instance needs"); ax.set_ylabel("solve rate")
    ax.set_title(f"density {rho:.2f}  (dashed = the gate that defines L*)")
axes[0].legend(loc="lower left", ncol=2, fontsize=9)
fig.tight_layout()
print("done")

## 6. Comparison to the previous best on the same eval

The previous best at N=48 leak-free is **finding 33's L=6, 0.6153 on this same n=1500 pool** (job `50099945`,
10-29) -- same instances, same protocol, so the comparison below is paired and exact, not a cross-run
estimate. The table also carries the leaked-order N=48 ladder for context, but those rows sit on a **different
pool** (`shuffle_adj` is in `INSTANCE_KEYS`): only the protocol is shared, so they are quoted and not tested.

In [ ]:
prev = "L=6"
best = pool.loc[pool.solve_rate.idxmax(), "cell"]
x, y = solved[best], solved[prev]
won, lost = int((x & ~y).sum()), int((~x & y).sum())
print(f"previous best on this eval : {prev}  {y.mean():.4f}   (finding 33, job 50099945)")
print(f"new best on this eval      : {best}  {x.mean():.4f}   (this pair of jobs)")
print(f"paired delta               : {x.mean() - y.mean():+.4f}   discordant {won}-{lost}   "
      f"McNemar exact p = {mcnemar_exact(won, lost):.3g}")
print()
print("L* at N=48, by task and MLP width (all at rho=0.60, the density the gate is read on):")
print("  leaked-order,   ff=1024 : L* = 4    (finding 14, 80000 instances)")
print("  leaked-order,   ff=256  : L* = 6    (finding 14)")
print("  LEAK-FREE,      ff=1024 : L* in (8, 12]   <- this notebook, at 3x the data")
print("                            (= 8 at the easier rho=0.75, where the deepest bin is T=4)")
print()
print("so removing the adjacency-order leak costs at least FOUR rungs of depth at N=48,")
print("against the two rungs finding 33 could prove with the ladder still open at the top.")

## 7. Pre-registrations, scored

From the header of `scripts/slurm/sudoku2_n48_deep_lstar.sbatch`, fixed before either job landed.

- **(a) L=12 clears the 0.90 gate in every rho=0.60 bin, so L\* is finally defined.** *Confirmed* -- 6 of 6.
- **(b) L=8 lands in [0.70, 0.84] on the shared pool.** *Missed on the high side* -- 0.8660. The interval was
  built by interpolating finding 33's L=6 (0.6153) toward finding 23's N=24 ceiling, and that interpolation
  was too pessimistic.
- **(c) L\*(T) rises with T at whichever cell sits on the gate.** *Confirmed at L=8* -- the three bins it
  misses are the three deepest and the trend statistic is reported above. Note this is the pre-registration
  finding 22 could only confirm marginally (z=-1.92, p=0.055) and finding 33 could not test at all.
  Unpre-registered and stronger: **L=12 trends too** (z above), despite clearing every bin, so the result
  does not depend on reading a single cell that happens to straddle the threshold.
- **(d) the format controls stay clean, so the gaps are deduction and not grammar.** Scored in section 2.

## 8. Limitations

1. **One seed per cell.** The whole ladder is seed 0. Finding 37 has just shown, on the SAE side of this
   project, that a second seed can move a layer-indexed readout by one rung; nothing here rules that out for
   L\*, and the honest reading of "(8, 12]" is "(8, 12] at this seed".
2. **The bracket is three rungs wide.** L=9, L=10 and L=11 are unmeasured, so L\* could be any of 9..12. One
   more cell at L=10 halves the bracket; that is the cheapest next job.
3. **Only one cell straddles the gate.** Every cell except L=3 (pinned at zero) does trend downward in T,
   L=12 included, which is more than the pre-registration asked for -- but the *crossing point*, T=5/T=6,
   is read off L=8's six bins alone, and those bins are not a monotone staircase: L=8 scores 0.775 at T=7
   (n=129) and 0.857 at T=8 (n=70). The trend is solid; the exact crossing bin is not.
4. **Fast phase only** (finding 34). Every bin here is above the removal floor, so "reasoning depth" means
   the log N regime; nothing in this notebook speaks to the slow phase.
5. **ff=1024 throughout.** Finding 14 showed L\* moves when the MLP narrows, so every number here is
   "at ff=1024" and the leak-free width factor at N=48 is still unmeasured.